In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

Device: cuda


In [ ]:

MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2470, 0.2435, 0.2616)

train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

train_aug  = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=train_tf)
train_eval = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=eval_tf)
test_set   = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=eval_tf)

CLASSES = train_aug.classes
targets = np.array(train_aug.targets)

print("Train:", len(train_aug), "| Test:", len(test_set))
print("Classes:", CLASSES)
print("Samples per class:", np.bincount(targets))
x, y = train_eval[0]
print("Image shape:", x.shape, "| label:", CLASSES[y])

100%|██████████| 170M/170M [26:33<00:00, 107kB/s]


Train: 50000 | Test: 10000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Samples per class: [5000 5000 5000 5000 5000 5000 5000 5000 5000 5000]
Image shape: torch.Size([3, 32, 32]) | label: frog


In [ ]:
from torchvision.models import resnet18

def make_resnet18(num_classes: int = 10) -> nn.Module:
    model = resnet18(weights=None, num_classes=num_classes)
    # CIFAR stem: 3x3 conv, stride 1, no max-pool
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model

model = make_resnet18().to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params / 1e6:.2f}M")

dummy = torch.randn(2, 3, 32, 32, device=device)
print("Output shape:", model(dummy).shape)

Parameters: 11.17M
Output shape: torch.Size([2, 10])


In [ ]:
BATCH_SIZE = 128

def make_loader(dataset, shuffle: bool, batch_size: int = BATCH_SIZE):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=2, pin_memory=True)

def train_model(model, dataset, epochs=30, lr=0.1, weight_decay=5e-4, verbose=True):
    loader = make_loader(dataset, shuffle=True)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                          weight_decay=weight_decay, nesterov=True)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * len(loader))
    scaler = torch.amp.GradScaler("cuda")

    for epoch in range(epochs):
        model.train()
        total, correct, loss_sum = 0, 0, 0.0
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                out = model(x)
                loss = F.cross_entropy(out, y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()

            loss_sum += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)

        if verbose:
            print(f"Epoch {epoch+1:2d}/{epochs} | loss {loss_sum/total:.3f} | train acc {correct/total:.3f}")
    return model

@torch.no_grad()
def evaluate(model, dataset) -> float:
    model.eval()
    loader = make_loader(dataset, shuffle=False, batch_size=512)
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(x)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total

In [ ]:
set_seed(0)
quick = make_resnet18().to(device)
train_model(quick, Subset(train_aug, range(5000)), epochs=1)
print("Test acc:", evaluate(quick, test_set))

Epoch  1/1 | loss 3.510 | train acc 0.126
Test acc: 0.1597


In [ ]:
SPLIT_SEED = 123
rng = np.random.default_rng(SPLIT_SEED)
all_idx = np.arange(len(train_aug))

# 1. Random forgetting
FORGET_FRAC = 0.10
forget_idx_rand = np.sort(rng.choice(all_idx, size=int(FORGET_FRAC * len(all_idx)), replace=False))
retain_idx_rand = np.setdiff1d(all_idx, forget_idx_rand)

# 2. Class forgetting (
FORGET_CLASS = 3                              # let's forget cat
forget_idx_cls = np.where(targets == FORGET_CLASS)[0]
retain_idx_cls = np.where(targets != FORGET_CLASS)[0]

# For class forgetting, we also split the TEST set
test_targets = np.array(test_set.targets)
test_forget_idx_cls = np.where(test_targets == FORGET_CLASS)[0]
test_retain_idx_cls = np.where(test_targets != FORGET_CLASS)[0]

np.savez("splits.npz",
         forget_idx_rand=forget_idx_rand, retain_idx_rand=retain_idx_rand,
         forget_idx_cls=forget_idx_cls, retain_idx_cls=retain_idx_cls,
         test_forget_idx_cls=test_forget_idx_cls, test_retain_idx_cls=test_retain_idx_cls)

In [ ]:
for name, f, r in [("random", forget_idx_rand, retain_idx_rand),
                   ("class",  forget_idx_cls,  retain_idx_cls)]:
    assert len(np.intersect1d(f, r)) == 0
    assert len(f) + len(r) == len(all_idx)
    print(f"{name:6s} | |D_f| = {len(f):5d} | |D_r| = {len(r):5d}")

print("Random D_f, samples per class:", np.bincount(targets[forget_idx_rand], minlength=10))
print("Class D_f, labels:", np.unique(targets[forget_idx_cls]), "=", CLASSES[FORGET_CLASS])
print("Test split for class scenario:", len(test_forget_idx_cls), "forget /", len(test_retain_idx_cls), "retain")

random | |D_f| =  5000 | |D_r| = 45000
class  | |D_f| =  5000 | |D_r| = 45000
Random D_f, samples per class: [475 537 473 460 516 543 503 486 490 517]
Class D_f, labels: [3] = cat
Test split for class scenario: 1000 forget / 9000 retain


In [ ]:
import os, time

os.makedirs("checkpoints", exist_ok=True)

set_seed(0)
M_o = make_resnet18().to(device)

t0 = time.time()
train_model(M_o, train_aug, epochs=30)
time_o = time.time() - t0

torch.save(M_o.state_dict(), "checkpoints/M_o.pt")
print(f"Training time: {time_o / 60:.1f} min")

Epoch  1/30 | loss 1.896 | train acc 0.316
Epoch  2/30 | loss 1.285 | train acc 0.528
Epoch  3/30 | loss 0.958 | train acc 0.659
Epoch  4/30 | loss 0.745 | train acc 0.740
Epoch  5/30 | loss 0.626 | train acc 0.784
Epoch  6/30 | loss 0.563 | train acc 0.806
Epoch  7/30 | loss 0.510 | train acc 0.825
Epoch  8/30 | loss 0.475 | train acc 0.837
Epoch  9/30 | loss 0.442 | train acc 0.849
Epoch 10/30 | loss 0.414 | train acc 0.858
Epoch 11/30 | loss 0.387 | train acc 0.866
Epoch 12/30 | loss 0.368 | train acc 0.874
Epoch 13/30 | loss 0.343 | train acc 0.880
Epoch 14/30 | loss 0.327 | train acc 0.888
Epoch 15/30 | loss 0.300 | train acc 0.898
Epoch 16/30 | loss 0.275 | train acc 0.907
Epoch 17/30 | loss 0.256 | train acc 0.913
Epoch 18/30 | loss 0.232 | train acc 0.920
Epoch 19/30 | loss 0.209 | train acc 0.928
Epoch 20/30 | loss 0.178 | train acc 0.939
Epoch 21/30 | loss 0.155 | train acc 0.947
Epoch 22/30 | loss 0.129 | train acc 0.956
Epoch 23/30 | loss 0.105 | train acc 0.965
Epoch 24/30

In [ ]:
def eval_func(model, scenario: str) -> dict:
    if scenario == "random":
        return {
            "retain": evaluate(model, Subset(train_eval, retain_idx_rand)),
            "forget": evaluate(model, Subset(train_eval, forget_idx_rand)),
            "test":   evaluate(model, test_set),
        }
    elif scenario == "class":
        return {
            "retain":      evaluate(model, Subset(train_eval, retain_idx_cls)),
            "forget":      evaluate(model, Subset(train_eval, forget_idx_cls)),
            "test_retain": evaluate(model, Subset(test_set, test_retain_idx_cls)),
            "test_forget": evaluate(model, Subset(test_set, test_forget_idx_cls)),
        }

def show(name, res):
    print(f"{name:12s} | " + " | ".join(f"{k} {v*100:5.1f}%" for k, v in res.items()))

show("M_o random", eval_func(M_o, "random"))
show("M_o class",  eval_func(M_o, "class"))

M_o random   | retain  99.7% | forget  99.6% | test  93.5%
M_o class    | retain  99.8% | forget  99.1% | test_retain  94.6% | test_forget  83.6%


In [ ]:
import json

SEEDS = [1, 2, 3]
retain_sets = {"random": retain_idx_rand, "class": retain_idx_cls}

results_r = {"random": [], "class": []}
times_r   = {"random": [], "class": []}

for scenario, retain_idx in retain_sets.items():
    for seed in SEEDS:
        path = f"checkpoints/M_r_{scenario}_seed{seed}.pt"
        set_seed(seed)
        model = make_resnet18().to(device)

        if os.path.exists(path):
            model.load_state_dict(torch.load(path, map_location=device))
            print(f"Loaded {path}")
        else:
            print(f"Training M_r | {scenario} | seed {seed} ...")
            t0 = time.time()
            train_model(model, Subset(train_aug, retain_idx), epochs=30, verbose=False)
            times_r[scenario].append(time.time() - t0)
            torch.save(model.state_dict(), path)

        res = eval_func(model, scenario)
        results_r[scenario].append(res)
        show(f"M_r {scenario[:4]} s{seed}", res)

        del model
        torch.cuda.empty_cache()

with open("checkpoints/results_r.json", "w") as f:
    json.dump({"results": results_r, "times": times_r}, f)

Loaded checkpoints/M_r_random_seed1.pt
M_r rand s1  | retain  99.7% | forget  93.1% | test  93.4%
Training M_r | random | seed 2 ...
M_r rand s2  | retain  99.7% | forget  93.1% | test  93.2%
Training M_r | random | seed 3 ...
M_r rand s3  | retain  99.7% | forget  93.3% | test  93.3%
Training M_r | class | seed 1 ...
M_r clas s1  | retain  99.9% | forget   0.0% | test_retain  95.2% | test_forget   0.0%
Training M_r | class | seed 2 ...
M_r clas s2  | retain  99.8% | forget   0.0% | test_retain  94.9% | test_forget   0.0%
Training M_r | class | seed 3 ...
M_r clas s3  | retain  99.7% | forget   0.0% | test_retain  94.7% | test_forget   0.0%


In [ ]:
for scenario in ["random", "class"]:
    runs = results_r[scenario]
    summary = " | ".join(
        f"{k} {np.mean([r[k] for r in runs])*100:5.1f} ± {np.std([r[k] for r in runs])*100:.1f}"
        for k in runs[0]
    )
    print(f"M_r {scenario:6s} | {summary}")
    if times_r[scenario]:
        print(f"           mean training time: {np.mean(times_r[scenario]) / 60:.1f} min")

M_r random | retain  99.7 ± 0.0 | forget  93.2 ± 0.1 | test  93.3 ± 0.1
           mean training time: 10.3 min
M_r class  | retain  99.8 ± 0.1 | forget   0.0 ± 0.0 | test_retain  94.9 ± 0.2 | test_forget   0.0 ± 0.0
           mean training time: 10.3 min


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

@torch.no_grad()
def get_losses(model, dataset) -> np.ndarray:
    model.eval()
    loader = make_loader(dataset, shuffle=False, batch_size=512)
    losses = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x)
        losses.append(F.cross_entropy(logits.float(), y, reduction="none").cpu())
    return torch.cat(losses).numpy()

def mia_score(model, scenario: str, seed: int = 0) -> dict:
    if scenario == "random":
        retain_idx, forget_idx = retain_idx_rand, forget_idx_rand
        nonmember_set = test_set
    else:
        retain_idx, forget_idx = retain_idx_cls, forget_idx_cls
        nonmember_set = Subset(test_set, test_retain_idx_cls)   # no cats on either side

    rng = np.random.default_rng(seed)
    member_idx = rng.choice(retain_idx, size=len(nonmember_set), replace=False)

    feat = lambda l: np.log(l + 1e-8).reshape(-1, 1)
    X_in  = feat(get_losses(model, Subset(train_eval, member_idx)))
    X_out = feat(get_losses(model, nonmember_set))
    X = np.vstack([X_in, X_out])
    y = np.concatenate([np.ones(len(X_in)), np.zeros(len(X_out))])

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    attack = LogisticRegression().fit(X_tr, y_tr)

    X_f = feat(get_losses(model, Subset(train_eval, forget_idx)))
    return {
        "attack_acc":    attack.score(X_te, y_te),
        "forget_member": attack.predict(X_f).mean(),
        "test_member":   attack.predict(X_te[y_te == 0]).mean(),
    }

In [ ]:
def load_model(path):
    m = make_resnet18().to(device)
    m.load_state_dict(torch.load(path, map_location=device))
    return m

for scenario in ["random", "class"]:
    show(f"MIA M_o {scenario[:4]}", mia_score(M_o, scenario))
    for seed in SEEDS:
        m = load_model(f"checkpoints/M_r_{scenario}_seed{seed}.pt")
        show(f"MIA M_r {scenario[:4]} s{seed}", mia_score(m, scenario))
        del m

MIA M_o rand | attack_acc  54.1% | forget_member  59.9% | test_member  54.1%
MIA M_r rand s1 | attack_acc  53.1% | forget_member  53.0% | test_member  54.4%
MIA M_r rand s2 | attack_acc  52.5% | forget_member  53.3% | test_member  53.6%
MIA M_r rand s3 | attack_acc  53.1% | forget_member  53.6% | test_member  53.6%
MIA M_o clas | attack_acc  52.1% | forget_member  35.2% | test_member  54.4%
MIA M_r clas s1 | attack_acc  52.1% | forget_member   0.0% | test_member  54.6%
MIA M_r clas s2 | attack_acc  52.1% | forget_member   0.0% | test_member  53.1%
MIA M_r clas s3 | attack_acc  52.2% | forget_member   0.0% | test_member  54.1%


In [ ]:
class Ensemble(nn.Module):
    def __init__(self, models):
        super().__init__()
        self.models = nn.ModuleList(models)

    def forward(self, x):
        probs = torch.stack([F.softmax(m(x).float(), dim=1) for m in self.models]).mean(0)
        return torch.log(probs + 1e-12)   # log-probabilities

In [ ]:
N_SHARDS = 5
shard_rng = np.random.default_rng(7)
perm = shard_rng.permutation(len(train_aug))
shards = np.array_split(perm, N_SHARDS)          # 5 disjoint arrays of 10,000 indices
np.savez("checkpoints/sisa_shards.npz", *shards)

def train_shard(idx, seed):
    set_seed(seed)
    m = make_resnet18().to(device)
    train_model(m, Subset(train_aug, idx), epochs=30, verbose=False)
    return m

sisa_models = []
t0 = time.time()
for k, idx in enumerate(shards):
    path = f"checkpoints/sisa_o_shard{k}.pt"
    if os.path.exists(path):
        m = load_model(path)
    else:
        print(f"Training shard {k} ({len(idx)} samples) ...")
        m = train_shard(idx, seed=100 + k)
        torch.save(m.state_dict(), path)
    print(f"  shard {k} test acc: {evaluate(m, test_set)*100:.1f}%")
    sisa_models.append(m)
time_sisa_o = time.time() - t0

sisa_o = Ensemble(sisa_models)
show("SISA_o rand",  eval_func(sisa_o, "random"))
show("SISA_o class", eval_func(sisa_o, "class"))

  shard 0 test acc: 73.5%
  shard 1 test acc: 74.8%
  shard 2 test acc: 76.8%
  shard 3 test acc: 71.3%
  shard 4 test acc: 79.6%
SISA_o rand  | retain  83.5% | forget  83.5% | test  80.4%
SISA_o class | retain  85.3% | forget  67.3% | test_retain  82.4% | test_forget  62.2%


In [ ]:
def sisa_unlearn(forget_idx, scenario):
    new_models, retrained = [], []
    t0 = time.time()
    for k, (idx, m) in enumerate(zip(shards, sisa_models)):
        hit = np.isin(idx, forget_idx)
        if hit.any():
            path = f"checkpoints/sisa_{scenario}_shard{k}.pt"
            if os.path.exists(path):
                m_new = load_model(path)
            else:
                print(f"Retraining shard {k}: removing {hit.sum()} samples ...")
                m_new = train_shard(idx[~hit], seed=100 + k)
                torch.save(m_new.state_dict(), path)
            new_models.append(m_new)
            retrained.append(k)
        else:
            new_models.append(m)          # untouched shard: reuse as is
    return Ensemble(new_models), retrained, time.time() - t0

for scenario, f_idx in [("random", forget_idx_rand), ("class", forget_idx_cls)]:
    sisa_u, retrained, t = sisa_unlearn(f_idx, scenario)
    print(f"{scenario}: retrained shards {retrained} in {t/60:.1f} min")
    show(f"SISA_u {scenario[:4]}", eval_func(sisa_u, scenario))
    show(f"MIA SISA_u {scenario[:4]}", mia_score(sisa_u, scenario))

Retraining shard 0: removing 976 samples ...
Retraining shard 1: removing 1017 samples ...
Retraining shard 2: removing 1058 samples ...
Retraining shard 3: removing 958 samples ...
Retraining shard 4: removing 991 samples ...
random: retrained shards [0, 1, 2, 3, 4] in 10.8 min
SISA_u rand  | retain  78.7% | forget  77.2% | test  76.1%
MIA SISA_u rand | attack_acc  50.7% | forget_member  40.8% | test_member  40.5%
Retraining shard 0: removing 1044 samples ...
Retraining shard 1: removing 1015 samples ...
Retraining shard 2: removing 996 samples ...
Retraining shard 3: removing 974 samples ...
Retraining shard 4: removing 971 samples ...
class: retrained shards [0, 1, 2, 3, 4] in 10.7 min
SISA_u clas  | retain  86.9% | forget   0.0% | test_retain  83.6% | test_forget   0.0%
MIA SISA_u clas | attack_acc  50.6% | forget_member   0.0% | test_member  41.9%


In [ ]:
import copy
from torch.utils.data import Dataset, ConcatDataset

class RelabeledDataset(Dataset):
    def __init__(self, base, indices, new_labels):
        self.base, self.indices, self.new_labels = base, indices, new_labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        x, _ = self.base[self.indices[i]]      #dropping the true label
        return x, int(self.new_labels[i])

def random_wrong_labels(idx, seed=0):
    rng = np.random.default_rng(seed)
    true = targets[idx]
    return (true + rng.integers(1, 10, size=len(idx))) % 10

In [ ]:
def data_mod_unlearn(scenario, epochs=3, lr=0.01, seed=0):
    forget_idx = forget_idx_rand if scenario == "random" else forget_idx_cls
    retain_idx = retain_idx_rand if scenario == "random" else retain_idx_cls

    unlearn_set = ConcatDataset([
        RelabeledDataset(train_aug, forget_idx, random_wrong_labels(forget_idx, seed)),
        Subset(train_aug, retain_idx),
    ])

    set_seed(seed)
    model = copy.deepcopy(M_o)            # not modifying M_o itself
    t0 = time.time()
    train_model(model, unlearn_set, epochs=epochs, lr=lr, verbose=False)
    return model, time.time() - t0

for scenario in ["random", "class"]:
    m_u, t = data_mod_unlearn(scenario)
    print(f"{scenario}: {t/60:.1f} min")
    show(f"DataMod {scenario[:4]}", eval_func(m_u, scenario))
    show(f"MIA DataMod {scenario[:4]}", mia_score(m_u, scenario))
    del m_u

random: 1.2 min
DataMod rand | retain  99.6% | forget  97.9% | test  93.1%
MIA DataMod rand | attack_acc  54.7% | forget_member  45.4% | test_member  53.2%
class: 1.1 min
DataMod clas | retain  99.9% | forget   0.0% | test_retain  95.4% | test_forget   0.0%
MIA DataMod clas | attack_acc  51.8% | forget_member   0.0% | test_member  52.0%


In [ ]:
for epochs, lr in [(1, 0.01), (3, 0.01), (1, 0.001), (3, 0.001)]:
    m_u, t = data_mod_unlearn("random", epochs=epochs, lr=lr)
    show(f"ep{epochs} lr{lr}", eval_func(m_u, "random"))
    del m_u

ep1 lr0.01   | retain  99.4% | forget  98.4% | test  93.1%
ep3 lr0.01   | retain  99.6% | forget  97.9% | test  93.1%
ep1 lr0.001  | retain  98.9% | forget  98.6% | test  91.7%
ep3 lr0.001  | retain  99.6% | forget  99.3% | test  93.3%


In [ ]:
dm_rand, _ = data_mod_unlearn("random")
for name, m in [("M_o", M_o),
                ("M_r s1", load_model("checkpoints/M_r_random_seed1.pt")),
                ("DataMod", dm_rand)]:
    lf = get_losses(m, Subset(train_eval, forget_idx_rand))
    lt = get_losses(m, test_set)
    print(f"{name:8s} | median loss D_f {np.median(lf):.4f} | test {np.median(lt):.4f}")

M_o      | median loss D_f 0.0002 | test 0.0003
M_r s1   | median loss D_f 0.0003 | test 0.0003
DataMod  | median loss D_f 0.1184 | test 0.1050


In [ ]:
for name, m in [("M_o", M_o),
                ("M_r s1", load_model("checkpoints/M_r_random_seed1.pt")),
                ("DataMod", dm_rand)]:
    lf = get_losses(m, Subset(train_eval, forget_idx_rand))
    lt = get_losses(m, test_set)
    p = [50, 90, 95, 99]
    print(f"{name:8s} | D_f  " + "  ".join(f"p{q}={v:.4f}" for q, v in zip(p, np.percentile(lf, p))))
    print(f"{'':8s} | test " + "  ".join(f"p{q}={v:.4f}" for q, v in zip(p, np.percentile(lt, p))))

M_o      | D_f  p50=0.0002  p90=0.0096  p95=0.0312  p99=0.2917
         | test p50=0.0003  p90=0.2167  p95=1.3117  p99=5.2789
M_r s1   | D_f  p50=0.0003  p90=0.2814  p95=1.5307  p99=5.7346
         | test p50=0.0003  p90=0.2470  p95=1.4246  p99=5.2976
DataMod  | D_f  p50=0.1184  p90=0.3577  p95=0.6021  p99=1.4584
         | test p50=0.1050  p90=0.6431  p95=1.3505  p99=3.2568


In [ ]:
def reset_layers(model, names=("layer4", "fc")):
    for name in names:
        for m in getattr(model, name).modules():
            if hasattr(m, "reset_parameters"):
                m.reset_parameters()
    return model

def reset_unlearn(scenario, names=("layer4", "fc"), epochs=5, lr=0.05, seed=0):
    retain_idx = retain_idx_rand if scenario == "random" else retain_idx_cls
    set_seed(seed)
    model = reset_layers(copy.deepcopy(M_o), names)
    t0 = time.time()
    train_model(model, Subset(train_aug, retain_idx), epochs=epochs, lr=lr, verbose=False)
    return model, time.time() - t0

for scenario in ["random", "class"]:
    m_u, t = reset_unlearn(scenario)
    print(f"{scenario}: {t/60:.1f} min")
    show(f"Reset {scenario[:4]}", eval_func(m_u, scenario))
    show(f"MIA Reset {scenario[:4]}", mia_score(m_u, scenario))
    del m_u

random: 1.8 min
Reset rand   | retain  98.3% | forget  95.2% | test  92.5%
MIA Reset rand | attack_acc  52.3% | forget_member  49.7% | test_member  48.5%
class: 1.7 min
Reset clas   | retain  98.4% | forget   0.0% | test_retain  94.3% | test_forget   0.0%
MIA Reset clas | attack_acc  50.2% | forget_member   0.0% | test_member  49.7%


In [ ]:
for names in [("fc",), ("layer4", "fc"), ("layer3", "layer4", "fc")]:
    m_u, t = reset_unlearn("random", names=names)
    show("+".join(names), eval_func(m_u, "random"))
    show("MIA", mia_score(m_u, "random"))
    del m_u

fc           | retain  99.0% | forget  96.5% | test  93.0%
MIA          | attack_acc  52.8% | forget_member  55.4% | test_member  53.2%
layer4+fc    | retain  98.3% | forget  95.2% | test  92.5%
MIA          | attack_acc  52.3% | forget_member  49.7% | test_member  48.5%
layer3+layer4+fc | retain  92.6% | forget  89.5% | test  89.1%
MIA          | attack_acc  51.0% | forget_member  48.6% | test_member  48.4%


In [ ]:
reset_rand, _ = reset_unlearn("random")
lf = get_losses(reset_rand, Subset(train_eval, forget_idx_rand))
lt = get_losses(reset_rand, test_set)
p = [50, 90, 95, 99]
print("Reset D_f  " + "  ".join(f"p{q}={v:.4f}" for q, v in zip(p, np.percentile(lf, p))))
print("Reset test " + "  ".join(f"p{q}={v:.4f}" for q, v in zip(p, np.percentile(lt, p))))

Reset D_f  p50=0.0001  p90=0.1431  p95=0.7424  p99=3.4751
Reset test p50=0.0001  p90=0.3907  p95=1.5708  p99=5.3750


In [ ]:
def neggrad_plus(scenario, epochs=2, lr=0.01, alpha=0.95, seed=0):
    forget_idx = forget_idx_rand if scenario == "random" else forget_idx_cls
    retain_idx = retain_idx_rand if scenario == "random" else retain_idx_cls

    set_seed(seed)
    model = copy.deepcopy(M_o)
    retain_loader = make_loader(Subset(train_aug, retain_idx), shuffle=True)
    forget_loader = DataLoader(Subset(train_aug, forget_idx), batch_size=BATCH_SIZE,
                           shuffle=True, num_workers=0)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scaler = torch.amp.GradScaler("cuda")

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        forget_iter = iter(forget_loader)
        for x_r, y_r in retain_loader:
            try:
                x_f, y_f = next(forget_iter)
            except StopIteration:
                forget_iter = iter(forget_loader)
                x_f, y_f = next(forget_iter)

            x_r, y_r = x_r.to(device, non_blocking=True), y_r.to(device, non_blocking=True)
            x_f, y_f = x_f.to(device, non_blocking=True), y_f.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                loss_r = F.cross_entropy(model(x_r), y_r)
                loss_f = F.cross_entropy(model(x_f), y_f)
                loss = alpha * loss_r - (1 - alpha) * loss_f
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
    return model, time.time() - t0

In [ ]:
for scenario in ["random", "class"]:
    for alpha in [0.99, 0.95, 0.9]:
        m_u, t = neggrad_plus(scenario, alpha=alpha)
        show(f"NG+ {scenario[:4]} a={alpha} ({t/60:.1f}m)", eval_func(m_u, scenario))
        show("   MIA", mia_score(m_u, scenario))
        del m_u

NG+ rand a=0.99 (2.0m) | retain  99.7% | forget  99.6% | test  93.6%
   MIA       | attack_acc  54.1% | forget_member  59.7% | test_member  53.6%
NG+ rand a=0.95 (2.0m) | retain  99.7% | forget  99.5% | test  93.5%
   MIA       | attack_acc  54.1% | forget_member  59.5% | test_member  53.4%
NG+ rand a=0.9 (2.0m) | retain  99.7% | forget  99.4% | test  93.5%
   MIA       | attack_acc  54.1% | forget_member  59.4% | test_member  53.5%
NG+ clas a=0.99 (2.0m) | retain  99.1% | forget   8.0% | test_retain  94.2% | test_forget   7.1%
   MIA       | attack_acc  51.3% | forget_member   0.0% | test_member  56.0%
NG+ clas a=0.95 (2.0m) | retain  96.6% | forget   0.0% | test_retain  91.5% | test_forget   0.0%
   MIA       | attack_acc  50.6% | forget_member   0.0% | test_member  59.6%
NG+ clas a=0.9 (2.0m) | retain  92.7% | forget   0.0% | test_retain  88.6% | test_forget   0.0%
   MIA       | attack_acc  51.3% | forget_member   0.0% | test_member  64.6%


In [ ]:
for alpha in [0.8, 0.7, 0.5]:
    m_u, t = neggrad_plus("random", alpha=alpha)
    show(f"NG+ rand a={alpha}", eval_func(m_u, "random"))
    show("   MIA", mia_score(m_u, "random"))
    del m_u

for alpha in [0.98, 0.97]:
    m_u, t = neggrad_plus("class", alpha=alpha)
    show(f"NG+ clas a={alpha}", eval_func(m_u, "class"))
    show("   MIA", mia_score(m_u, "class"))
    del m_u

NG+ rand a=0.8 | retain  99.6% | forget  99.1% | test  93.2%
   MIA       | attack_acc  53.6% | forget_member  59.0% | test_member  54.0%
NG+ rand a=0.7 | retain  99.1% | forget  97.4% | test  92.5%
   MIA       | attack_acc  53.0% | forget_member  59.3% | test_member  54.2%
NG+ rand a=0.5 | retain  35.9% | forget  30.5% | test  34.2%
   MIA       | attack_acc  50.7% | forget_member  26.6% | test_member  30.6%
NG+ clas a=0.98 | retain  98.6% | forget   0.1% | test_retain  93.5% | test_forget   0.2%
   MIA       | attack_acc  51.1% | forget_member   0.0% | test_member  55.6%
NG+ clas a=0.97 | retain  97.9% | forget   0.0% | test_retain  92.7% | test_forget   0.0%
   MIA       | attack_acc  50.6% | forget_member   0.0% | test_member  56.5%


In [ ]:
import pandas as pd
from scipy.stats import ks_2samp

def collect(model, scenario):
    if scenario == "random":
        forget_idx, unseen = forget_idx_rand, test_set
    else:
        forget_idx, unseen = forget_idx_cls, Subset(test_set, test_forget_idx_cls)
    mia = mia_score(model, scenario)
    return {
        "acc":     eval_func(model, scenario),
        "mia_gap": mia["forget_member"] - mia["test_member"],
        "loss_f":  get_losses(model, Subset(train_eval, forget_idx)),
        "loss_u":  get_losses(model, unseen),
    }

def ks(a, b):
    return ks_2samp(a, b).statistic

def make_row(name, st, ref_stats, t, t_ref):
    row = {"method": name}
    row.update({k: round(v * 100, 1) for k, v in st["acc"].items()})
    row["MIA gap"]          = round(st["mia_gap"] * 100, 1)
    row["KS to M_r"]        = round(np.mean([ks(st["loss_f"], r["loss_f"]) for r in ref_stats]), 3)
    row["KS D_f vs unseen"] = round(ks(st["loss_f"], st["loss_u"]), 3)
    row["time (min)"]       = round(t / 60, 1) if t else None
    row["speedup"]          = round(t_ref / t, 1) if t else None
    return row

def mr_row(ref_stats, t_ref):
    row = {"method": "M_r (3 seeds)"}
    row.update({k: round(np.mean([r["acc"][k] for r in ref_stats]) * 100, 1) for k in ref_stats[0]["acc"]})
    pairs = [ks(a["loss_f"], b["loss_f"]) for i, a in enumerate(ref_stats) for b in ref_stats[i + 1:]]
    row["MIA gap"]          = round(np.mean([r["mia_gap"] for r in ref_stats]) * 100, 1)
    row["KS to M_r"]        = round(np.mean(pairs), 3)
    row["KS D_f vs unseen"] = round(np.mean([ks(r["loss_f"], r["loss_u"]) for r in ref_stats]), 3)
    row["time (min)"]       = round(t_ref / 60, 1)
    row["speedup"]          = 1.0
    return row

In [ ]:
with open("checkpoints/results_r.json") as f:
    time_r = {s: np.mean(v) for s, v in json.load(f)["times"].items()}

SISA_TIME = {"random": 10.8 * 60, "class": 10.7 * 60}
ALPHA     = {"random": 0.7, "class": 0.98}
forget_sets = {"random": forget_idx_rand, "class": forget_idx_cls}

tables = {}
for scenario in ["random", "class"]:
    ref_stats = []
    for s in SEEDS:
        m = load_model(f"checkpoints/M_r_{scenario}_seed{s}.pt")
        ref_stats.append(collect(m, scenario))
        del m
    t_ref = time_r[scenario]

    rows = [mr_row(ref_stats, t_ref),
            make_row("M_o", collect(M_o, scenario), ref_stats, None, t_ref)]

    builders = {
        "SISA":     lambda: (sisa_unlearn(forget_sets[scenario], scenario)[0], SISA_TIME[scenario]),
        "DataMod":  lambda: data_mod_unlearn(scenario),
        "Reset":    lambda: reset_unlearn(scenario),
        "NegGrad+": lambda: neggrad_plus(scenario, alpha=ALPHA[scenario]),
    }
    for name, build in builders.items():
        model, t = build()
        rows.append(make_row(name, collect(model, scenario), ref_stats, t, t_ref))
        del model
        torch.cuda.empty_cache()
        print(f"done: {scenario} / {name}")

    tables[scenario] = pd.DataFrame(rows).set_index("method")
    tables[scenario].to_csv(f"results_{scenario}.csv")

for s, df in tables.items():
    print(f"\n=== {s} scenario ===")
    display(df)

done: random / SISA
done: random / DataMod
done: random / Reset
done: random / NegGrad+
done: class / SISA
done: class / DataMod
done: class / Reset
done: class / NegGrad+

=== random scenario ===


,retain,forget,test,MIA gap,KS to M_r,KS D_f vs unseen,time (min),speedup
method,,,,,,,,
M_r (3 seeds),99.7,93.2,93.3,-0.5,0.029,0.011,10.3,1.0
M_o,99.7,99.6,93.5,5.7,0.128,0.116,NaN,NaN
SISA,78.7,77.2,76.1,0.3,0.660,0.017,10.8,1.0
DataMod,99.5,99.5,93.2,3.5,0.780,0.098,1.2,8.9
Reset,79.6,80.2,76.9,0.0,0.807,0.030,1.7,6.0
NegGrad+,99.1,97.4,92.5,5.1,0.075,0.083,2.0,5.2



=== class scenario ===


,retain,forget,test_retain,test_forget,MIA gap,KS to M_r,KS D_f vs unseen,time (min),speedup
method,,,,,,,,,
M_r (3 seeds),99.8,0.0,94.9,0.0,-54.0,0.069,0.024,10.3,1.0
M_o,99.8,99.1,94.6,83.6,-19.2,1.000,0.229,NaN,NaN
SISA,86.9,0.0,83.6,0.0,-41.9,0.551,0.018,10.7,1.0
DataMod,99.7,0.0,95.2,0.0,-52.5,1.000,0.157,1.2,8.9
Reset,83.0,0.0,80.3,0.0,-48.5,0.990,0.022,1.7,5.9
NegGrad+,98.6,0.1,93.5,0.2,-55.6,0.185,0.147,2.0,5.2


In [ ]:
import inspect
print(inspect.signature(data_mod_unlearn))
print(inspect.signature(reset_unlearn))
print(inspect.signature(reset_layers))
print(inspect.signature(train_model))
print(inspect.signature(make_loader), "| BATCH_SIZE =", BATCH_SIZE)

(scenario, epochs=3, lr=0.01, seed=0)
(scenario, names=('layer4', 'fc'), epochs=5, lr=0.05, seed=0)
(model, names=('layer4', 'fc'))
(model, dataset, epochs=30, lr=0.1, weight_decay=0.0005, verbose=True)
(dataset, shuffle: bool, batch_size: int = 128) | BATCH_SIZE = 128


In [ ]:
m, t = reset_unlearn("random")
show("Reset now", eval_func(m, "random")); del m

m, t = data_mod_unlearn("random")
show("DataMod now", eval_func(m, "random")); del m

Reset now    | retain  79.6% | forget  80.2% | test  76.9%
DataMod now  | retain  99.5% | forget  99.5% | test  93.2%


In [ ]:
for f in [make_loader, train_model, reset_layers, reset_unlearn, data_mod_unlearn, random_wrong_labels]:
    print(inspect.getsource(f))
    print("-" * 60)

def make_loader(dataset, shuffle: bool, batch_size: int = BATCH_SIZE):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=2, pin_memory=True)

------------------------------------------------------------
def train_model(model, dataset, epochs=30, lr=0.1, weight_decay=5e-4, verbose=True):
    loader = make_loader(dataset, shuffle=True)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                          weight_decay=weight_decay, nesterov=True)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs * len(loader))
    scaler = torch.amp.GradScaler("cuda")

    for epoch in range(epochs):
        model.train()
        total, correct, loss_sum = 0, 0, 0.0
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
             

In [ ]:
ref = torch.load("checkpoints/M_o.pt", map_location=device)
print("Weights identical to checkpoint:",
      all(torch.equal(v, ref[k]) for k, v in M_o.state_dict().items()))

frozen = [n for n, p in M_o.named_parameters() if not p.requires_grad]
print(f"Frozen parameters: {len(frozen)} of {sum(1 for _ in M_o.parameters())}")
print(frozen[:3], "...", frozen[-3:])

Weights identical to checkpoint: True
Frozen parameters: 0 of 62
[] ... []


In [ ]:
saved = np.load("splits.npz")
print("Splits unchanged:", all(np.array_equal(globals()[k], saved[k]) for k in saved.files))
print("Targets unchanged:", np.array_equal(targets, np.array(train_aug.targets)),
      np.array_equal(targets, np.array(train_eval.targets)))
print(train_aug.transform)

Splits unchanged: True
Targets unchanged: True True
Compose(
    RandomCrop(size=(32, 32), padding=4)
    RandomHorizontalFlip(p=0.5)
    ToTensor()
    Normalize(mean=(0.4914, 0.4822, 0.4465), std=(0.247, 0.2435, 0.2616))
)


In [ ]:
M_o_fresh = load_model("checkpoints/M_o.pt")

set_seed(0)
m = reset_layers(copy.deepcopy(M_o_fresh))
train_model(m, Subset(train_aug, retain_idx_rand), epochs=5, lr=0.05, verbose=True)
show("Reset (fresh M_o)", eval_func(m, "random"))
del m

Epoch  1/5 | loss 1.414 | train acc 0.634
Epoch  2/5 | loss 0.988 | train acc 0.737
Epoch  3/5 | loss 0.880 | train acc 0.756
Epoch  4/5 | loss 0.828 | train acc 0.772
Epoch  5/5 | loss 0.805 | train acc 0.780
Reset (fresh M_o) | retain  79.6% | forget  80.2% | test  76.9%
